In [ ]:
# ==============================
# Clustering of the dataset
# ==============================

import h5py
import numpy as np
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)   # Inputs: [vc, fz, eps_r, eps_a, z, ri]
    Y = f["Y"][:].astype(np.float32)   # Outputs: [Sa, Sz]

print("Dataset loaded")
print("X shape:", X.shape, "Y shape:", Y.shape)

# ==============================
# DBSCAN clustering on outputs
# ==============================
tol_1d = 0.01
tol_2d = 0.01

def compute_clusters(Y_sub, name, tol):
    clustering = DBSCAN(eps=tol, min_samples=2).fit(Y_sub)
    labels = clustering.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"\n{name}: Found {n_clusters} clusters (DBSCAN, tol={tol})")
    return labels

labels_sa = compute_clusters(Y[:, [0]], "Sa only", tol_1d)
labels_sz = compute_clusters(Y[:, [1]], "Sz only", tol_1d)
labels_sa_sz = compute_clusters(Y, "Sa+Sz together", tol_2d)

# ==============================
# 1) Density of outputs (Sa vs Sz)
# ==============================
plt.figure(figsize=(6,5), dpi=180)

plt.hist2d(Y[:,0], Y[:,1], bins=50, cmap='jet')
plt.colorbar(label='Number of points')

plt.xlabel(r'$S_a$ ($\mu$m)')
plt.ylabel(r'$S_z$ ($\mu$m)')
plt.title('Density of Outputs ($S_a$, $S_z$)')

plt.tight_layout()
plt.show()

# ==============================
# 2) Input space colored by output clusters (vc vs fz) ⭐
# ==============================
vc = X[:,0]
fz = X[:,1]

plt.figure(figsize=(6,5), dpi=180)

sc = plt.scatter(
    vc,
    fz,
    c=labels_sa_sz,
    cmap='tab20',
    s=6,
    alpha=0.7
)

plt.xlabel(r'$v_c$ (m/min)')
plt.ylabel(r'$f_z$ (mm/tooth)')
plt.title('Input Space colored by Output Clusters')

cbar = plt.colorbar(sc)
cbar.set_label('Output cluster ID')

plt.tight_layout()
plt.show()

# ==============================
# 3) Multiplicity: inputs per output cluster ⭐
# ==============================
cluster_sizes = [
    np.sum(labels_sa_sz == lbl)
    for lbl in set(labels_sa_sz) if lbl != -1
]

# cluster_sizes = [
#     np.sum(labels_sa_sz == lbl)
#     for lbl in set(labels_sa_sz) if lbl != -1
# ]
plt.figure(figsize=(6,4), dpi=180)

plt.hist(cluster_sizes, bins=30)

plt.xlabel('Number of inputs per output cluster')
plt.ylabel('Count')
plt.title('Multiplicity of Inputs Mapping to Same Output')

plt.tight_layout()
plt.show()

# ==============================
# Print example clusters
# ==============================
unique_labels = sorted(set(labels_sa_sz))

for idx_count, lbl in enumerate(unique_labels):
    if lbl == -1 or idx_count % 20 != 0:
        continue

    idx = np.where(labels_sa_sz == lbl)[0]

    print(f"\nCluster {lbl} -> {len(idx)} points")
    print("Inputs (first 5):\n", X[idx][:5])
    print("Outputs (first 5):\n", Y[idx][:5])

In [ ]:
import h5py
import numpy as np
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)   # Inputs: [vc, fz, eps_r, eps_a, z, ri]
    Y = f["Y"][:].astype(np.float32)   # Outputs: [Sa, Sz]

print("Dataset loaded")
print("X shape:", X.shape, "Y shape:", Y.shape)

# ==============================
# Function to compute clusters
# ==============================
def compute_clusters(Y_sub, tol):
    clustering = DBSCAN(eps=tol, min_samples=2).fit(Y_sub)
    labels = clustering.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    return n_clusters

# ==============================
# Sweep tolerances and record cluster counts
# ==============================
tolerances = np.linspace(0.001, 0.1, 50)  # from 0.001 µm to 0.05 µm
clusters_sa = []
clusters_sz = []
clusters_sa_sz = []

for tol in tolerances:
    n_sa = compute_clusters(Y[:, [0]], tol)
    n_sz = compute_clusters(Y[:, [1]], tol)
    n_sa_sz = compute_clusters(Y, tol)

    clusters_sa.append(n_sa)
    clusters_sz.append(n_sz)
    clusters_sa_sz.append(n_sa_sz)

# ==============================
# Plot cluster counts vs tolerance
# ==============================
plt.figure(figsize=(6,4), dpi=180)
plt.plot(tolerances, clusters_sa, marker='o', label='$S_a$ only')
plt.plot(tolerances, clusters_sz, marker='s', label='$S_z$ only')
plt.plot(tolerances, clusters_sa_sz, marker='^', label='$S_a$+$S_z$ together')

plt.xlabel('DBSCAN Tolerance (µm)')
plt.ylabel('Number of Clusters')
plt.title('DBSCAN Clusters vs Tolerance')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# Dense-aware Exploratory Plots with controllable point size
# ==============================

import h5py
import numpy as np
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.colors import Normalize


# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)      # Inputs: [vc, fz, eps_r, eps_a, z, ri]
    Y = f["Y"][:].astype(np.float32)      # Outputs already: [Sa, Sz]

print("Dataset loaded")
print("X shape:", X.shape, "Y shape (Sa, Sz):", Y.shape)

# ------------------------------

# ===== PLOT STYLE CONTROL (PUT HERE) =====
plt.rcParams.update({
    'figure.dpi': 180,
    'savefig.dpi': 400,
    'font.size': 13,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})
# Adjustable point size
point_size_2d = 10     # 2D scatter
point_size_3d = 5      # 3D scatter
max_points_2d = 5000   # max points per 2D scatter
max_points_3d = 10000  # max points per 3D scatter

input_names = [
    r'$v_c$ (m/min)', r'$f_z$ (mm/tooth)', r'$\varepsilon_r$ (mm)',
    r'$\varepsilon_a$ (mm)', r'$z$', r'$r_i$'
]

Sa = Y[:,0]
Sz = Y[:,1]
N = len(Sa)
np.random.seed(0)

# Function to subsample points in dense areas
def subsample_dense(X_data, Y_data, max_points):
    """Subsample points if dataset is too large"""
    n = len(Y_data)
    if n <= max_points:
        return np.arange(n)
    else:
        return np.random.choice(n, max_points, replace=False)

# ==============================
# 1) Each input vs each output (2D scatter)
# ==============================
for i, name in enumerate(input_names):

    # Subsample for dense regions
    idx = subsample_dense(X[:, i], Sa, max_points_2d)
    X_plot = X[idx, i]
    Sa_plot = Sa[idx]
    Sz_plot = Sz[idx]

    # Input vs Sa
    fig, ax = plt.subplots(figsize=(6,4))
    norm_sz = Normalize(vmin=np.min(Sz_plot), vmax=np.max(Sz_plot))
    sc = ax.scatter(X_plot, Sa_plot,
                    s=point_size_2d, c=Sz_plot, cmap='jet',
                    alpha=0.6, edgecolors='none', norm=norm_sz, rasterized=True)
    ax.set_xlabel(name)
    ax.set_ylabel(r'$S_a$ ($\mu$m)')
    ax.set_title(f'{name} vs $S_a$ (color=$S_z$)')
    cbar = plt.colorbar(sc)
    cbar.set_label(r'$S_z$ ($\mu$m)')
    plt.tight_layout()
    plt.show()

    # Input vs Sz
    fig, ax = plt.subplots(figsize=(6,4))
    norm_sa = Normalize(vmin=np.min(Sa_plot), vmax=np.max(Sa_plot))
    sc = ax.scatter(X_plot, Sz_plot,
                    s=point_size_2d, c=Sa_plot, cmap='jet',
                    alpha=0.6, edgecolors='none', norm=norm_sa, rasterized=True)
    ax.set_xlabel(name)
    ax.set_ylabel(r'$S_z$ ($\mu$m)')
    ax.set_title(f'{name} vs $S_z$ (color=$S_a$)')
    cbar = plt.colorbar(sc)
    cbar.set_label(r'$S_a$ ($\mu$m)')
    plt.tight_layout()
    plt.show()

# ==============================
# 2) Each input vs BOTH outputs (3D scatter)
# ==============================
for i, name in enumerate(input_names):
    idx = subsample_dense(X[:, i], Sa, max_points_3d)
    X_plot = X[idx, i]
    Sa_plot_3d = Sa[idx]
    Sz_plot_3d = Sz[idx]

    fig = plt.figure(figsize=(6,5))
    ax = fig.add_subplot(111, projection='3d')

    norm_input = Normalize(vmin=np.min(X_plot), vmax=np.max(X_plot))
    sc = ax.scatter(X_plot, Sa_plot_3d, Sz_plot_3d,
                    c=X_plot, cmap='jet',
                    s=point_size_3d, alpha=0.6,
                    edgecolors='none', norm=norm_input, rasterized=True)

    ax.set_xlabel(name)
    ax.set_ylabel(r'$S_a$')
    ax.set_zlabel(r'$S_z$')
    plt.title(f'3D relation: {name} vs $S_a$ vs $S_z$')
    cbar = plt.colorbar(sc)
    cbar.set_label(name)
    plt.tight_layout()
    plt.show()

# ==============================
# 3) Global overview 3D (Sa, Sz, vc)
# ==============================
idx = subsample_dense(X[:,0], Sa, max_points_3d)
Sa_plot_3d = Sa[idx]
Sz_plot_3d = Sz[idx]
vc_plot_3d = X[idx,0]
fz_plot_3d = X[idx,1]

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

norm_fz = Normalize(vmin=np.min(fz_plot_3d), vmax=np.max(fz_plot_3d))
sc = ax.scatter(Sa_plot_3d, Sz_plot_3d, vc_plot_3d,
                c=fz_plot_3d, cmap='jet',
                s=point_size_3d, alpha=0.6,
                edgecolors='none', norm=norm_fz, rasterized=True)

ax.set_xlabel(r'$S_a$ ($\mu$m)')
ax.set_ylabel(r'$S_z$')
ax.set_zlabel(r'$v_c$')
plt.title('Global relation: outputs vs inputs (color=$f_z$)')
cbar = plt.colorbar(sc)
cbar.set_label(r'$f_z$')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 3D plots: Input1 vs Input2 vs Sa, color=Sz
# z-label safe from colorbar
# ==============================
from matplotlib.colors import Normalize

# Adjustable parameters
point_size_3d = 5
max_points_3d = 8000
font_size_title = 13
font_size_label = 13
font_size_ticks = 10
figure_dpi = 180
savefig_dpi = 400
alpha_scatter = 0.6
label_pad = 8  # z-label distance increased

plt.rcParams.update({
    'figure.dpi': figure_dpi,
    'savefig.dpi': savefig_dpi,
    'font.size': font_size_label,
    'axes.titlesize': font_size_title,
    'axes.labelsize': font_size_label,
    'xtick.labelsize': font_size_ticks,
    'ytick.labelsize': font_size_ticks
})

def subsample_points(n_points, max_points):
    return np.arange(n_points) if n_points <= max_points else np.random.choice(n_points, max_points, replace=False)

# input_names = [r'$v_c$', r'$f_z$', r'$\varepsilon_r$', r'$\varepsilon_a$', r'$z$', r'$r_i$']

input_names = [
    r'$v_c$ (m/min)', r'$f_z$ (mm/tooth)', r'$\varepsilon_r$ (mm)',
    r'$\varepsilon_a$ (mm)', r'$z$', r'$r_i$'
]

Sa = Y[:,0]
Sz = Y[:,1]
np.random.seed(0)

for i in range(len(input_names)):
    for j in range(i+1, len(input_names)):
        x_data = X[:,i]
        y_data = X[:,j]
        z_data = Sa
        color_data = Sz

        idx = subsample_points(len(z_data), max_points_3d)
        x_plot, y_plot, z_plot, c_plot = x_data[idx], y_data[idx], z_data[idx], color_data[idx]

        fig = plt.figure(figsize=(9,7), dpi=figure_dpi)
        ax = fig.add_subplot(111, projection='3d')

        norm_c = Normalize(vmin=np.min(c_plot), vmax=np.max(c_plot))
        sc = ax.scatter(x_plot, y_plot, z_plot,
                        c=c_plot, cmap='jet', s=point_size_3d,
                        alpha=alpha_scatter, edgecolors='none', norm=norm_c, rasterized=True)

        # Axis labels with vertical rotation and extra padding
        ax.set_xlabel(input_names[i], labelpad=label_pad, rotation=0)
        ax.set_ylabel(input_names[j], labelpad=label_pad, rotation=0)
        ax.set_zlabel(r'$S_a$ ($\mu$m)', labelpad=label_pad - 7, rotation=270)  # extra pad for z-label

        plt.title(f'{input_names[i]} vs {input_names[j]} vs $S_a$ (color=$S_z$)')

        ax.invert_xaxis()

        # Colorbar moved away from z-axis
        cbar = fig.colorbar(sc, ax=ax, pad=0.1, shrink=0.5, location='right')  # pad increased
        cbar.set_label(r'$S_z$ ($\mu$m)')

        # Adjust layout
        plt.subplots_adjust(left=0.15, right=0.85, top=0.9, bottom=0.1)
        plt.show()